# DATE2 实验5：Prefetch Window × Weight Chunk敏感性


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve().parent if Path.cwd().name=='fig' else Path.cwd().resolve()
OUT=ROOT/'outputs/DATE2';FIG=ROOT/'fig/DATE2';FIG.mkdir(parents=True,exist_ok=True)
PUBLIC=['Static-NoPF','Static-NaivePF','Dynamic-NoPF','Dynamic-NaivePF','MemDomain']
FINAL_INTERNAL='MemDomain-'+'Safe'
def public_rows(frame):
    q=frame[frame.baseline.isin(['Static-NoPF','Static-NaivePF','Dynamic-NoPF',
                                 FINAL_INTERNAL])].copy()
    q['baseline']=q.baseline.replace({FINAL_INTERNAL:'MemDomain'})
    assert set(q.baseline)==set(PUBLIC)
    return q

rows=[]
for path in sorted((OUT/'window_chunk').glob('w*_c*.csv')):
    w,c=map(int,re.search(r'w(\d+)_c(\d+)',path.stem).groups())
    q=public_rows(pd.read_csv(path));q['window']=w;q['chunk_tiles']=c;rows.append(q)
d=pd.concat(rows,ignore_index=True)
mem=d[d.baseline=='MemDomain'];base=d[d.baseline=='Static-NoPF']
p=mem.pivot(index='window',columns='chunk_tiles',values='total_cycles')
b=base.pivot(index='window',columns='chunk_tiles',values='total_cycles')
gain=(1-p/b)*100
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,data,title,cmap in ((axes[0],p,'MemDomain total cycles','viridis'),
                          (axes[1],gain,'Gain vs Static-NoPF (%)','YlGn')):
    im=ax.imshow(data,aspect='auto',cmap=cmap);ax.set_xticks(range(len(data.columns)),data.columns)
    ax.set_yticks(range(len(data.index)),data.index);ax.set_xlabel('Chunk tiles')
    ax.set_ylabel('Prefetch window');ax.set_title(title);fig.colorbar(im,ax=ax)
plt.tight_layout();plt.savefig(FIG/'exp5_public_sensitivity.pdf',bbox_inches='tight');plt.show()
display(gain)
